# 第15章 高频数据与市场微结构 —— 配套代码

对应正文 `book/part4/15-high-frequency.md`。高频数据**体量大、含微结构噪声**。
本章用**合成日内数据**离线演示日内模式、买卖价差、已实现波动率与微结构噪声。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装。")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fds import set_chinese_font

set_chinese_font()
rng = np.random.default_rng(15)

## 1. 合成日内分钟数据

一个约 4 小时交易日的分钟价格（随机游走）+ 典型「U 形」日内成交量。

In [ ]:
n = 240
ret = rng.normal(0, 0.0008, n)
price = 100 * np.exp(np.cumsum(ret))
t = np.linspace(0, 1, n)
volume = (1.5 - np.sin(np.pi * t)) * 1000 + rng.normal(0, 60, n)
bar = pd.DataFrame({'minute': range(n), 'price': price, 'volume': volume})
print(bar.head())

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
a1.plot(bar['minute'], bar['price'], color='#2E5A88'); a1.set_ylabel('价格'); a1.set_title('日内分钟价格')
a2.bar(bar['minute'], bar['volume'], color='#888888', width=1.0)
a2.set_ylabel('成交量'); a2.set_xlabel('交易分钟'); a2.set_title('日内成交量（U 形）')
plt.tight_layout(); plt.show()

## 2. 买卖价差与微结构噪声（bid-ask bounce）

观测价在买价/卖价之间「跳动」，使高频收益出现**人为的负自相关**，并非真实价格波动。

In [ ]:
spread = 0.02                       # 固定买卖价差
mid = price
side = rng.choice([-1, 1], n)        # 每笔成交在买/卖一侧
trade = mid + side * spread / 2      # 观测成交价
obs_ret = np.diff(np.log(trade))
lag1_autocorr = np.corrcoef(obs_ret[:-1], obs_ret[1:])[0, 1]
print(f'观测成交价收益的一阶自相关 = {lag1_autocorr:.3f}（显著为负即 bid-ask bounce）')

## 3. 已实现波动率与波动率特征图

已实现波动率（RV）= 高频收益平方和。抽样越密，**微结构噪声越放大 RV**——这就是著名的「波动率特征图」。

In [ ]:
m = 23400                                   # 一个 6.5 小时交易日的秒数
efficient = np.cumsum(rng.normal(0, 1e-4, m))   # 有效价格（对数）
observed = efficient + rng.normal(0, 2e-4, m)   # 叠加微结构噪声
intervals = [1, 2, 5, 10, 30, 60, 120, 300, 600]
rv = [float(np.sum(np.diff(observed[::k]) ** 2)) for k in intervals]
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(intervals, rv, 'o-', color='#C0504D')
ax.set_xscale('log')
ax.set_title('波动率特征图：抽样间隔越小，RV 被噪声放大')
ax.set_xlabel('抽样间隔（秒，对数轴）'); ax.set_ylabel('已实现方差 RV')
plt.tight_layout(); plt.show()
print('实务常用 5 分钟（300 秒）抽样，在噪声与信息间折中。')

## 4. 成交量加权平均价（VWAP）

VWAP 是执行算法的常用基准：按成交量加权的平均成交价。

In [ ]:
vwap = float((bar['price'] * bar['volume']).sum() / bar['volume'].sum())
twap = float(bar['price'].mean())
print(f'VWAP = {vwap:.4f}  TWAP（时间加权）= {twap:.4f}')

## 习题参考解答

**习题1：把买卖价差扩大一倍，观察 bid-ask bounce 的一阶自相关如何变化。**

In [ ]:
for sp in (0.02, 0.04):
    tr = price + rng.choice([-1, 1], n) * sp / 2
    r = np.diff(np.log(tr))
    print(f'价差={sp}: 一阶自相关 = {np.corrcoef(r[:-1], r[1:])[0, 1]:.3f}')

**习题2：用不同抽样间隔估计 RV，验证 5 分钟抽样比逐秒更接近有效价格的真实方差。**

In [ ]:
true_var = float(np.sum(np.diff(efficient) ** 2))
for k in (1, 300):
    est = float(np.sum(np.diff(observed[::k]) ** 2))
    print(f'间隔{k}秒: RV={est:.4f}  （有效价格真实方差≈{true_var:.4f}）')